# Linked Lists

Notes and runnable examples on linked lists — the one classic structure Python **doesn't** give you built-in. The interesting questions here aren't just *how* you build one, but **why CPython avoids them in favor of arrays**, and where it quietly uses them anyway (`deque`, `OrderedDict`, `lru_cache`).

**Contents**

1. **What it is** — nodes, pointers, the flipped trade-off
2. **A singly linked list from scratch**
3. **The reality check** — why arrays usually win
4. **Doubly linked lists** — the real O(1) splice
5. **Where CPython actually uses linked lists**
6. **When to use what**
7. **Complexity cheat-sheet**

## 1. What it is

A linked list stores each element in its own **node**: a value plus a pointer to the **next** node (and, in a *doubly* linked list, the **previous** one too). Nodes are scattered across the heap and chained by reference — there's no contiguous block and no index arithmetic.

That flips the array trade-off on its head:

- **Insert / delete at a known position is O(1)** — relink a couple of pointers, nothing shifts.
- **Random access is O(n)** — to reach element *i* you must walk the chain from the head.

So linked lists shine when you splice a lot at positions you already hold, and lose badly when you index.

## 2. A singly linked list from scratch

Each node points to the next; the list keeps a reference to the **head**. Prepending is O(1) (just make a new head); appending is O(n) unless you also track a **tail** pointer. We put `__slots__` on the node to shrink it — more on why in the next section.

In [1]:
class Node:
    __slots__ = ("value", "next")
    def __init__(self, value, nxt=None):
        self.value = value
        self.next = nxt

class SinglyLinkedList:
    def __init__(self):
        self.head = None
        self._size = 0

    def prepend(self, value):           # O(1) — new node becomes the head
        self.head = Node(value, self.head)
        self._size += 1

    def __iter__(self):                 # walk the chain from the head
        node = self.head
        while node is not None:
            yield node.value
            node = node.next

    def __len__(self):
        return self._size

    def find(self, value):              # O(n) — no random access
        for i, v in enumerate(self):
            if v == value:
                return i
        return -1

ll = SinglyLinkedList()
for x in [3, 2, 1]:
    ll.prepend(x)                       # each prepend is O(1)
print("contents:", list(ll))           # [1, 2, 3]
print("len     :", len(ll))
print("find(2) :", ll.find(2))


contents: [1, 2, 3]
len     : 3
find(2) : 1


## 3. The reality check — why arrays usually win

The textbook sells linked lists on "O(1) insertion," but two things undercut that in real Python:

1. **You rarely already hold the node.** O(1) splice assumes you have a reference to the spot. *Finding* it is O(n) — so for "insert where *x* is" work the search dominates and a `list` is just as good.
2. **Nodes are expensive and scattered.** Every node is a separate heap object (its own header + two pointers), versus a list's single packed 8-byte pointer per element. That's more memory *and* worse cache locality — the CPU chases pointers all over the heap instead of streaming one contiguous block.

In [2]:
import sys, timeit

n = Node(42)
print("one node object :", sys.getsizeof(n), "bytes (+ the value object elsewhere)")
print("one list slot   :  8 bytes (one pointer, + the value object)")

N = 1_000_000
head = None
for i in range(N):
    head = Node(i, head)
arr = list(range(N))

def sum_linked():
    s, node = 0, head
    while node is not None:
        s += node.value
        node = node.next
    return s

def sum_array():
    s = 0
    for x in arr:
        s += x
    return s

t_ll  = timeit.timeit(sum_linked, number=3)
t_arr = timeit.timeit(sum_array,  number=3)
print(f"traverse linked list: {t_ll*1000:6.1f} ms")
print(f"traverse list       : {t_arr*1000:6.1f} ms  (~{t_ll/t_arr:.1f}x faster, both O(n))")


one node object : 48 bytes (+ the value object elsewhere)
one list slot   :  8 bytes (one pointer, + the value object)


traverse linked list:   44.9 ms
traverse list       :   36.0 ms  (~1.2x faster, both O(n))


Both traversals are O(n), but the array wins — and the gap is actually *understated* here: Python's per-step interpreter overhead dominates, muting the cache penalty. In a systems language (C, Rust) the contiguous array pulls much further ahead. This is exactly why CPython builds `list` on a contiguous array and ships **no** node-based linked list.

## 4. Doubly linked lists — the real O(1) splice

Add a `prev` pointer and you get a **doubly linked list**. Now you can delete a node in **true O(1)** *given a reference to it*, because you can reach both neighbors and relink them. This is the one place linked lists genuinely beat arrays — an array delete is always O(n) since everything after the gap shifts down.

In [3]:
class DNode:
    __slots__ = ("value", "prev", "next")
    def __init__(self, value):
        self.value = value
        self.prev = self.next = None

class DoublyLinkedList:
    def __init__(self):
        self.head = self.tail = None

    def append(self, value):            # O(1) thanks to the tail pointer
        node = DNode(value)
        node.prev = self.tail
        if self.tail:
            self.tail.next = node
        else:
            self.head = node
        self.tail = node
        return node                     # hand the node back so callers can splice it later

    def delete(self, node):             # O(1) — relink neighbors, nothing shifts
        if node.prev: node.prev.next = node.next
        else:         self.head = node.next
        if node.next: node.next.prev = node.prev
        else:         self.tail = node.prev

    def __iter__(self):
        node = self.head
        while node:
            yield node.value
            node = node.next

dll = DoublyLinkedList()
nodes = [dll.append(c) for c in "ABCDE"]
print("before:", list(dll))
dll.delete(nodes[2])                    # remove 'C' in O(1) — we already hold its node
print("after :", list(dll))


before: ['A', 'B', 'C', 'D', 'E']
after : ['A', 'B', 'D', 'E']


## 5. Where CPython actually uses linked lists

There's no public linked-list type, but the pattern is all over the standard library — used for exactly that O(1) relink:

- **`collections.deque`** is a doubly linked list of **fixed-size blocks** (arrays of ~64 elements), not single nodes. The blocks give O(1) push/pop at both ends *and* good cache locality — which is why it's the right tool for queue-like work.
- **`OrderedDict`** keeps a doubly linked list over its entries, so `move_to_end` and `popitem(last=...)` are O(1).
- **`functools.lru_cache`** uses a *circular* doubly linked list to track recency and evict the least-recently-used entry in O(1).

In [4]:
from collections import deque, OrderedDict

d = deque([1, 2, 3])
d.appendleft(0)                         # O(1) at the front
d.append(4)                             # O(1) at the back
print("deque:", d, "| popleft:", d.popleft(), "| pop:", d.pop())
d.rotate(1)
print("after rotate(1):", d)

od = OrderedDict.fromkeys("abcde")
od.move_to_end("a")                     # O(1) relink via the internal doubly linked list
print("OrderedDict order:", list(od))


deque: deque([1, 2, 3]) | popleft: 0 | pop: 4
after rotate(1): deque([3, 1, 2])
OrderedDict order: ['b', 'c', 'd', 'e', 'a']


## 6. When to use what

| You need... | Reach for | Why |
|---|---|---|
| An indexable, general-purpose sequence | `list` | contiguous, O(1) index, cache-friendly |
| O(1) push/pop at both ends | `collections.deque` | block-based DLL, no shifting |
| O(1) splice while holding node refs | a hand-rolled doubly linked list | true constant-time relink (rare in practice) |
| LRU / recency ordering | `functools.lru_cache` / `OrderedDict` | DLL-backed, O(1) move & evict |
| "A linked list" for its own sake | usually still a `list` | in Python the array almost always wins |

## 7. Complexity cheat-sheet

| Operation | Singly LL | Doubly LL | `list` | `deque` |
|---|:---:|:---:|:---:|:---:|
| Index access | O(n) | O(n) | O(1) | O(n) |
| Insert / delete at head | O(1) | O(1) | O(n) | O(1) |
| Insert / delete at tail | O(n)† | O(1) | O(1) amortized | O(1) |
| Delete given a node | O(n)‡ | O(1) | O(n) | — |
| Search by value | O(n) | O(n) | O(n) | O(n) |
| Memory per element | high (node obj) | higher (2 ptrs) | low (1 ptr) | low-ish (blocked) |

<sub>† O(1) if a tail pointer is maintained. &middot; ‡ a singly linked list needs the *predecessor* to relink, so it's O(1) only if you already hold that — doubly linked reaches both neighbors directly.</sub>